# 01 · Sentetik El Yazısı Verisi Üretimi
Türkçe-doğrulanmış el yazısı fontlarıyla biyoloji/tıp korpusundan satır görüntüleri üretir. Çıktı `veri/` (img + labels.jsonl) → 02 ve 03 defterlerinin girdisi.

**Çalışma zamanı:** GPU gerekmez (CPU yeterli). Drive'a bağlanıp çıktı orada saklanır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# sinav-okur paketini kur
!git clone https://github.com/KULLANICI/sinav-okur.git  # <-- kendi repo URL'nizle değiştirin || echo 'repo zaten var'
%cd sinav-okur
!pip install -q -e .

## Türkçe-uygun el yazısı fontlarını indir
Araştırmada tüm Türkçe karakterleri (çğıöşüİ) içerdiği cmap ile doğrulanmış OFL fontlar. `fontlari_dogrula` ile indirilen her font tekrar denetlenir; eksik olan atılır.

In [ ]:
import os, urllib.request
from sinav_okur.veri import ONERILEN_FONTLAR, fontlari_dogrula

# Google Fonts raw dizininden indir (font adı -> repo yolu)
FONT_URL = 'https://raw.githubusercontent.com/google/fonts/main/ofl/{slug}/{dosya}'
DOSYALAR = {
  'Caveat':'Caveat[wght].ttf','Caveat Brush':'CaveatBrush-Regular.ttf',
  'Dancing Script':'DancingScript[wght].ttf','Pacifico':'Pacifico-Regular.ttf',
  'Patrick Hand':'PatrickHand-Regular.ttf','Kalam':'Kalam-Regular.ttf',
  'Amatic SC':'AmaticSC-Regular.ttf','Indie Flower':'IndieFlower-Regular.ttf',
  'Marck Script':'MarckScript-Regular.ttf','Courgette':'Courgette-Regular.ttf',
  'Great Vibes':'GreatVibes-Regular.ttf','Sacramento':'Sacramento-Regular.ttf',
  'Bad Script':'BadScript-Regular.ttf','Merienda':'Merienda[wght].ttf',
  'Itim':'Itim-Regular.ttf',
}
os.makedirs('fontlar', exist_ok=True)
for ad in ONERILEN_FONTLAR:
    slug = ad.lower().replace(' ', '')
    url = FONT_URL.format(slug=slug, dosya=DOSYALAR[ad])
    hedef = f'fontlar/{DOSYALAR[ad]}'
    try:
        urllib.request.urlretrieve(url, hedef); print('✓', ad)
    except Exception as e:
        print('✗', ad, e)

In [ ]:
# İndirilen fontların Türkçe kapsamını doğrula
rapor = fontlari_dogrula('fontlar')
for yol, eksik in rapor.items():
    print('✓' if not eksik else '✗', yol.split('/')[-1], eksik[:8])

## Veri üret
`--ek-terimler` yerine burada kendi biyoloji/tıp terim listenizi ekleyebilirsiniz. Öğrenci başına ~30k satır iyi bir başlangıçtır; T4 gerekmez.

In [ ]:
from sinav_okur.veri import veri_uret

EK_TERIMLER = None  # örn: ['nükleik asit','organ nakli','antibiyotik']
sonuc = veri_uret('veri', 'fontlar', adet=30000, tohum=1, ek_terimler=EK_TERIMLER)
print(sonuc)

In [ ]:
# Örnek satırlara göz at
import json, matplotlib.pyplot as plt, matplotlib.image as mpimg
kayitlar = [json.loads(l) for l in open('veri/labels.jsonl')][:6]
fig, eks = plt.subplots(6,1, figsize=(9,8))
for e, k in zip(eks, kayitlar):
    e.imshow(mpimg.imread('veri/'+k['image']), cmap='gray'); e.set_title(k['text']); e.axis('off')
plt.tight_layout()

## Gerçek Türkçe el yazısını harmanla — T-H-E veri seti
Veri-avı araştırması (17 ajan) sonucu: doğrulanmış TEK gerçek Türkçe el yazısı
GÖRÜNTÜ verisi **T-H-E** (200 yazar, 28x28 karakter, Türkçe harfler dahil). HuggingFace'te
YOK — GitHub'dan git-lfs ile CSV gelir. Karakter düzeyi olduğundan `the_satir_uret`
gerçek glifleri biyoloji/tıp korpusuna dizerek GERÇEK-glifli satırlar üretir ve AYNI
`veri/` dizinine ekler (sentetik+gerçek tek hattan eğitilir).

> Lisans: T-H-E'de LICENSE yok → atıfla eğitim için (Ediboğlu Bartos ve ark.,
> Acta Polytechnica Hungarica 2020); veriyi yeniden dağıtmayın. Rakam/noktalama
> T-H-E'de yoktur; o glifleri içeren satırlar sentetik-font hattından gelir.
> Asıl gerçek kazanç kendi kâğıtlarınızın `sinav-okur disaaktar` flywheel'idir.

In [ ]:
# T-H-E'yi indir (git-lfs ŞART; yoksa sadece pointer iner)
!git lfs install
!git clone https://github.com/bartosgaye/thedataset 2>/dev/null || echo 'repo mevcut'
!cd thedataset && git lfs pull
import glob; print('CSV dosyaları:', glob.glob('thedataset/*.csv'))

In [ ]:
# Gerçek-glifli biyoloji/tıp satırları üret, sentetik veriyle AYNI dizine ekle
from sinav_okur.veri import the_satir_uret
import glob
csv_yol = sorted(glob.glob('thedataset/*.csv'))[0]  # version1..6; tam 78 sınıf
r = the_satir_uret(csv_yol, 'veri', adet=8000, tohum=7, onek='the_')
print(r)

In [ ]:
# Eval için (eğitime KATMA): sentetik olduğu doğrulanan turkish-ocr ayrı dizine
from sinav_okur.veri import hf_labels_uret
try:
    print(hf_labels_uret('turkish-ocr', 'eval_seti'))
except Exception as e:
    print('eval seti atlandı:', e)

In [ ]:
# Toplam karışım (sentetik font + T-H-E gerçek glif)
n = sum(1 for _ in open('veri/labels.jsonl'))
print('toplam eğitim satırı (sentetik + gerçek-glif):', n)

In [ ]:
# Drive'a arşivle
!zip -qr /content/drive/MyDrive/sinav_okur_veri.zip veri
print('Kaydedildi: MyDrive/sinav_okur_veri.zip')